# stochastic-rs — paths on the CPU and on CUDA, side by side

Samples a gallery of processes twice — once on the host, once on the GPU — and
plots the two next to each other, family by family: diffusions, short rates,
stochastic volatility, jumps, point processes, fractional processes,
subordinators and a conditional-variance model.

**What is and is not comparable.** The host draws from this crate's SIMD
ziggurat and the kernel hashes its own normals from `(path, step, seed)`, so
the two never produce the same *path* — comparing them point for point is
meaningless. What must agree is the **law**: the third panel of each row
overlays the two terminal distributions, and the table at the end puts the
means and spreads beside each other. A row whose paths look different but
whose histograms sit on top of each other is exactly right.

**Where a process falls back.** Some configurations exceed what the kernels
carry, and the process says so: `device_fallback()` returns the reason and
`device_ready()` is the absence of one. The gallery prints both, so a row
that ran on the host despite `device="cuda"` is visible rather than merely
slow.

Cells: 1 GPU · 2 Rust · 3 repository · 4 wheel (20-30 min) · 5 gallery ·
6 plots · 7 law table · 8 timings.

In [ ]:
# 1. The GPU and the CUDA toolkit. The driver's CUDA version (nvidia-smi, top right)
#    should be >= the toolkit's (nvcc): NVRTC emits PTX for the toolkit's version and
#    an older driver cannot JIT it (CUDA_ERROR_UNSUPPORTED_PTX_VERSION).
!nvidia-smi
!nvcc --version | tail -2

In [ ]:
# 2. Rust (stable, minimal profile). PATH is extended for every later cell.
import os
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
os.environ["CARGO_TERM_COLOR"] = "never"
!cargo --version && rustc --version

In [ ]:
# 3. The repository. REF is a branch, tag or commit; main is the default.
REF = "main"
!rm -rf stochastic-rs && git clone --quiet --depth 1 --branch {REF} https://github.com/rust-dd/stochastic-rs.git
%cd stochastic-rs
!git log --oneline -1

In [ ]:
# 4. The Python module with the CUDA back-end. Release build, 20-30 minutes on
#    Colab; maturin needs an explicit interpreter because Colab has no venv.
!pip -q install maturin numpy matplotlib
!maturin build --release --features cuda --interpreter python3 --out dist 2>&1 | tail -2
!pip -q install --force-reinstall --no-deps dist/*.whl

import importlib
import stochastic_rs as sr
sr = importlib.reload(sr)
print(sr.probe_device("cuda"))
print(sr.probe_device("cpu"))

In [ ]:
# 5. The gallery: what to sample, and how to reduce each one to a plottable
#    (paths, terminal values) pair. Two-component processes (Heston, Merton in
#    log space) hand back a tuple; the first component is the one plotted, the
#    second is kept for the law table.
import numpy as np
import stochastic_rs as sr

N = 512          # grid points per path
PATHS = 4_000    # paths per side; the plots draw the first few
DRAWN = 24       # paths per panel
SEED = 20260907

def one(x):
    """The first component of a process that returns several."""
    return x[0] if isinstance(x, tuple) else x

GALLERY = [
    # family, label, builder(device) -> process
    ("diffusion", "GBM  μ=0.05 σ=0.2",
     lambda d: sr.PyGbm(0.05, 0.2, N, x0=100.0, t=1.0, seed=SEED, device=d)),
    ("diffusion", "Ornstein-Uhlenbeck  θ=2 μ=0.04",
     lambda d: sr.PyOu(2.0, 0.04, 0.25, N, x0=0.12, t=1.0, seed=SEED, device=d)),
    ("diffusion", "CIR  θ=1.5 μ=0.05",
     lambda d: sr.PyCir(1.5, 0.05, 0.15, N, x0=0.05, t=1.0, seed=SEED, device=d)),
    ("diffusion", "Jacobi  α=0.9 β=3 (unit interval)",
     lambda d: sr.PyJacobi(0.9, 3.0, 0.5, N, x0=0.3, t=1.0, seed=SEED, device=d)),
    ("diffusion", "3/2 model  κ=4 μ=0.09",
     lambda d: sr.PyThreeHalf(4.0, 0.09, 0.8, N, x0=0.09, t=1.0, seed=SEED, device=d)),
    ("short rate", "Vasicek  θ=1.5 μ=0.04",
     lambda d: sr.PyVasicek(1.5, 0.04, 0.3, N, x0=0.04, t=1.0, seed=SEED, device=d)),
    ("short rate", "fractional Vasicek  H=0.7",
     lambda d: sr.PyFVasicek(0.7, 1.5, 0.04, 0.3, N, x0=0.04, t=1.0, seed=SEED, device=d)),
    ("volatility", "Heston (log price)  κ=2 ξ=0.5 ρ=-0.7",
     lambda d: sr.PyHestonLog(mu=0.03, kappa=2.0, theta=0.04, xi=0.5, rho=-0.7,
                              n=N, s0=100.0, v0=0.04, t=1.0, seed=SEED, device=d)),
    ("volatility", "SABR  α=0.4 β=0.5 ρ=-0.3",
     lambda d: sr.PySabr(0.4, 0.5, -0.3, N, f0=100.0, v0=0.2, t=1.0, seed=SEED, device=d)),
    ("jump", "Merton (log price)  λ=1.5",
     lambda d: sr.PyMjdLog(mu=0.03, sigma=0.2, lambda_=1.5, nu=-0.05, omega=0.15,
                           n=N, s0=100.0, t=1.0, seed=SEED, device=d)),
    ("jump", "variance gamma  θ=-0.2 ν=0.35",
     lambda d: sr.PyVg(-0.2, 0.4, 0.35, N, x0=0.0, t=1.0, seed=SEED, device=d)),
    ("jump", "normal inverse Gaussian  θ=0.15 κ=0.4",
     lambda d: sr.PyNig(0.15, 0.5, 0.4, N, x0=0.0, t=1.0, seed=SEED, device=d)),
    ("point process", "Poisson  λ=25",
     lambda d: sr.PyPoisson(25.0, n=N, seed=SEED, device=d)),
    ("point process", "Hawkes  μ=1 α=0.5 β=1.5 (event times)",
     lambda d: sr.PyHawkes(1.0, 0.5, 1.5, n=64, seed=SEED, device=d)),
    ("fractional", "fBm  H=0.7",
     lambda d: sr.PyFbm(0.7, N, t=1.0, seed=SEED, device=d)),
    ("fractional", "fGN  H=0.3 (increments)",
     lambda d: sr.PyFgn(0.3, N, t=1.0, seed=SEED, device=d)),
    ("subordinator", "α-stable subordinator  α=0.7",
     lambda d: sr.PyAlphaStableSubordinator(0.7, 0.8, N, x0=0.0, t=1.0, seed=SEED, device=d)),
    ("conditional variance", "GARCH(1,1)  ω=.15 α=.15 β=.7",
     lambda d: sr.PyGarch(0.15, [0.15], [0.7], N, seed=SEED, device=d)),
]

# What each process says about itself before anything runs: whether a kernel
# carries this configuration, and the reason when it does not.
print(f"{'process':44s} {'device':7s} reason")
for _, label, build in GALLERY:
    p = build("cuda")
    ready = p.device_ready()
    print(f"{label:44s} {'kernel' if ready else 'host':7s} {p.device_fallback() or ''}")

In [ ]:
# 6. The plots: CPU paths, CUDA paths, and the two terminal laws overlaid.
#    Left and centre share a y-scale so the eye compares dispersion, not axes.
import matplotlib.pyplot as plt

def sample_both(build):
    host = one(build("cpu").sample_par(PATHS))
    device = one(build("cuda").sample_par(PATHS))
    return np.asarray(host, dtype=float), np.asarray(device, dtype=float)

rows = len(GALLERY)
fig, axes = plt.subplots(rows, 3, figsize=(15, 2.6 * rows))
fig.suptitle("stochastic-rs — the same process on the host and on CUDA", y=1.001, fontsize=13)
samples = {}

for row, (family, label, build) in enumerate(GALLERY):
    host, device = sample_both(build)
    samples[label] = (host, device)
    grid = np.arange(host.shape[1])
    lo = min(host[:DRAWN].min(), device[:DRAWN].min())
    hi = max(host[:DRAWN].max(), device[:DRAWN].max())
    for col, (paths, name) in enumerate(((host, "CPU"), (device, "CUDA"))):
        ax = axes[row, col]
        ax.plot(grid, paths[:DRAWN].T, lw=0.6, alpha=0.75)
        ax.set_ylim(lo, hi)
        ax.set_title(f"{label} — {name}" if col == 0 else name, fontsize=9, loc="left")
        ax.tick_params(labelsize=7)
        if col == 0:
            ax.set_ylabel(family, fontsize=8)
    ax = axes[row, 2]
    edges = np.histogram_bin_edges(np.concatenate([host[:, -1], device[:, -1]]), bins=60)
    ax.hist(host[:, -1], bins=edges, histtype="step", lw=1.2, label="CPU", density=True)
    ax.hist(device[:, -1], bins=edges, histtype="step", lw=1.2, label="CUDA", density=True)
    ax.set_title("terminal law", fontsize=9, loc="left")
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7, frameon=False)

fig.tight_layout()
plt.show()

In [ ]:
# 7. The law table. The paths differ by construction; these numbers must not.
#    The band is the standard error of the difference of two independent
#    sample means (and of two spreads), so it is what the sample size allows
#    rather than a tolerance anyone chose.
print(f"{'process':44s} {'mean cpu':>11s} {'mean cuda':>11s} {'z':>6s} "
      f"{'sd cpu':>10s} {'sd cuda':>10s} {'z':>6s}")
for _, label, _ in GALLERY:
    host, device = samples[label]
    h, d = host[:, -1], device[:, -1]
    n = len(h)
    mean_z = (h.mean() - d.mean()) / np.sqrt(h.var(ddof=1) / n + d.var(ddof=1) / n)
    # The standard error of a sample standard deviation is sd·sqrt((κ−1)/4n).
    def spread_se(x):
        m2 = x.var()
        k = ((x - x.mean()) ** 4).mean() / (m2 * m2)
        return np.sqrt(m2) * np.sqrt(max(k - 1.0, 0.0) / (4 * len(x)))
    sd_z = (h.std() - d.std()) / np.hypot(spread_se(h), spread_se(d))
    print(f"{label:44s} {h.mean():11.5f} {d.mean():11.5f} {mean_z:6.1f} "
          f"{h.std():10.5f} {d.std():10.5f} {sd_z:6.1f}")
print("\nz is in standard errors; |z| under 5 is agreement, and a process that "
      "fell back to the host has z = 0 by construction.")

In [ ]:
# 8. Wall time per batch, host against device. The GPU wins where the batch is
#    large and the grid long; on a small batch the launch dominates and the CPU
#    is faster, which is worth seeing rather than assuming.
import time

def timed(build, device, m):
    build(device).sample_par(8)          # warm the context and the kernel cache
    start = time.perf_counter()
    one(build(device).sample_par(m))
    return (time.perf_counter() - start) * 1e3

print(f"{'process':44s} {'paths':>7s} {'cpu ms':>9s} {'cuda ms':>9s} {'speed-up':>9s}")
for _, label, build in GALLERY[:8]:
    for m in (256, 20_000):
        cpu_ms = timed(build, "cpu", m)
        gpu_ms = timed(build, "cuda", m)
        print(f"{label:44s} {m:7d} {cpu_ms:9.1f} {gpu_ms:9.1f} {cpu_ms / gpu_ms:8.1f}x")

## Reading the result

- **The path panels differ, and should.** Host and device draw different
  streams; only the law is shared. What to look for is the *shape* — the same
  drift, the same dispersion, the same boundary behaviour (a CIR that never
  goes negative, a Jacobi inside the unit interval, a subordinator that only
  increases).
- **The histograms should sit on top of each other**, and the table's `z`
  columns should stay inside ±5. A `z` of 30 is a real disagreement and worth
  an issue; a `z` of 4 on a heavy-tailed process (the α-stable subordinator,
  the 3/2 model) is the sample size talking.
- **A row marked `host` in cell 5** ran on the CPU on both sides despite
  `device="cuda"`, for the reason printed beside it — its `z` values are then
  a comparison of the host with itself.
- **Cell 8's small-batch rows are usually red for the GPU.** One launch has a
  fixed cost; the device pays off from a few thousand paths, and the crate
  chunks a batch too large for the device's memory into launches whose union
  is bit-identical to one.